<a href="https://colab.research.google.com/github/encoras/Introduction-to-OpenCV/blob/master/amber_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tar -xzvf archyvas.tar.gz

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm
from skimage import color

# === KONFIGŪRACIJA ===
DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/kevir_12_mazi"   # arba tavo kelias
CLASSES_TO_USE = ["NK01", "NK02","NK03","NK04","NK05","NK06","NK07","NK06","NK09","NK10","NK11", "NK12"]           # tik šviesiausia ir tamsiausia
PIXELS_PER_IMAGE = 8000                     # saugus skaičius (mažiau atminties)
N_CLUSTERS = 20

# === Surenkame pikselius tik iš dviejų klasių ===
print("Loading HSV pixels nk01 ir nk12...")

all_pixels = []

for class_name in CLASSES_TO_USE:
    class_dir = Path(DATA_ROOT) / class_name
    image_paths = list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpg"))

    print(f"→ {class_name}: {len(image_paths)} nuotraukų")

    for path in tqdm(image_paths[:20]):           # jei nori, gali apriboti
        img = cv2.imread(str(path))
        if img is None: continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        flat = img.reshape(-1, 3).astype(np.float32)

        # Imame atsitiktinius pikselius
        if len(flat) > PIXELS_PER_IMAGE:
            idx = np.random.choice(len(flat), PIXELS_PER_IMAGE, replace=False)
            flat = flat[idx]

        all_pixels.append(flat)

all_pixels = np.vstack(all_pixels)
print(f"Viso pikselių klasterizavimui: {len(all_pixels):,} ({all_pixels.nbytes / 1e9:.2f} GB)")

# === Klasterizavimas (greitas MiniBatchKMeans) ===
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=8192,
    random_state=42,
    n_init='auto'
)
kmeans.fit(all_pixels)

# Konvertuojame H iš 0-179 → 0-360
centers = kmeans.cluster_centers_.astype(int)


print("\nKlasterių centrai (H, S, V):")
for i, c in enumerate(centers):
    print(f"Klasteris {i:2d}:  H={c[0]:3d}  S={c[1]:3d}  V={c[2]:3d}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import colorsys
from skimage import color
# Tarkime, kad jau turi:
centers_hsv = kmeans.cluster_centers_   # shape (n_clusters, 3) → H,S,V
labels      = kmeans.labels_

# ────────────────────────────────────────────────────────────────
# 1. HSV → RGB konversija (0–1 diapazonas matplotlib'ui)
# ────────────────────────────────────────────────────────────────



cluster_rgb = (color.hsv2rgb(centers_hsv/255))  # naudok aukščiau pateiktą funkciją

cluster_sizes  = np.bincount(labels, minlength=len(centers_hsv))
total_pixels   = len(labels)
cluster_percent = (cluster_sizes / total_pixels) * 100

# Vizualizacija
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(1, 2, 1)
ax1.pie(
    cluster_percent,
    labels=[f'Cl {i}\n{perc:.1f}%' for i, perc in enumerate(cluster_percent)],
    autopct='%1.1f%%',
    colors=cluster_rgb,
    startangle=90,
    pctdistance=0.75
)
ax1.set_title("Clusters by area (%)")

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow([[rgb] for rgb in cluster_rgb], aspect='auto')

ax2.set_yticks(range(len(cluster_rgb)))
ax2.set_yticklabels([
    f"Cl {i:2d}  {perc:5.1f}%   RGB({int(r*255):3d},{int(g*255):3d},{int(b*255):3d})\n"
    f"H={(L):5.0f} S={(a):5.0f} V={(v):5.0f}"
    for i, (L,a,v), (r,g,b), perc in zip(range(len(centers_hsv)), centers_hsv, cluster_rgb, cluster_percent)
], fontsize=9)

ax2.set_xticks([])
ax2.set_title("Klasterių spalvos (RGB)")

plt.tight_layout()
plt.show()

In [ ]:
# ========================== 1. INSTALIACIJA IR IMPORTAI ==========================
!pip install opencv-python-headless scikit-learn pandas matplotlib seaborn -q

import cv2
import numpy as np
import pandas as pd
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab.patches import cv2_imshow

# ========================== 2. KONFIGŪRACIJA ==========================
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/kevir_12_mazi"   # ← pakeisk į savo kelią

# HSV ribos gintarui (labai gerai veikia ant šviesaus fono)

LOWER_AMBER = np.array([  0,  30,  15 ])
UPPER_AMBER = np.array([ 50, 255, 255 ])

# Morfologija
KERNEL = np.ones((5,5), np.uint8)

def segment_dark_amber(hsv):
    # Tamsiems gintarams – platesnis H, mažesnis V minimumas
    lower_dark = np.array([ 0,  70,  1 ])
    upper_dark = np.array([ 90, 255, 180 ])  # V iki 180, kad neperimtų šviesaus fono
    return cv2.inRange(hsv, lower_dark, upper_dark)

def segment_light_amber(hsv):
    lower_light = np.array([ 8,  60,  80 ])
    upper_light = np.array([ 63, 255, 255 ])
    return cv2.inRange(hsv, lower_light, upper_light)



# ========================== 3. SEGMENTAVIMO FUNKCIJA ==========================
def segment_amber(image_path):
    img = cv2.imread(str(image_path))
    img = cv2.GaussianBlur(img,(3,3),0)
    if img is None:
        return None, None



    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    #mask = cv2.inRange(hsv, LOWER_AMBER, UPPER_AMBER)
    # Sujungimas
    mask_dark  = segment_dark_amber(hsv)
    mask_light = segment_light_amber(hsv)
    mask = cv2.bitwise_or(mask_dark, mask_light)


    # Valymas
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)

    # Imame didžiausią kontūrą (kad nebūtų triukšmo)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    largest = max(contours, key=cv2.contourArea)
    mask_clean = np.zeros(mask.shape, np.uint8)
    cv2.drawContours(mask_clean, [largest], -1, 255, -1)

    return img, mask_clean

# ========================== 4. FEATURE EXTRACTION (LAB) ==========================
def extract_features(img, mask):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    masked_lab = lab[mask > 0]

    if len(masked_lab) < 100:  # per mažas gabalėlis
        return None

    mean_l, mean_a, mean_b = np.mean(masked_lab, axis=0)
    std_l,  std_a,  std_b  = np.std(masked_lab, axis=0)

    # Papildomi požymiai
    area = np.sum(mask > 0)
    perimeter = cv2.arcLength(cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0][0], True)
    circularity = 4 * np.pi * area / (perimeter * perimeter + 1e-6)

    return {
        'mean_L': mean_l, 'mean_A': mean_a, 'mean_B': mean_b,
        'std_L':  std_l,  'std_A':  std_a,  'std_B':  std_b,
        'area': area, 'circularity': circularity
    }

# ========================== 5. DUOMENŲ SURINKIMAS (TRAINING) ==========================
features_list = []
labels = []

print("✅ Renkame duomenis iš nk01-nk12...")

for nk_folder in sorted(Path(DATASET_PATH).glob("NK*")):
    class_name = nk_folder.name
    for img_path in nk_folder.glob("*.*"):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue

        img, mask = segment_amber(img_path)
        if img is None or mask is None:
            continue

        feat = extract_features(img, mask)
        if feat:
            features_list.append(feat)
            labels.append(class_name)

        # Parodome pirmus 3 pavyzdžius iš kiekvienos klasės
        if len(features_list) % 50 == 0:
            cv2_imshow(cv2.bitwise_and(img, img, mask=mask))

df = pd.DataFrame(features_list)
df['label'] = labels
print(f"✅ Surinkta {len(df)} pavyzdžių iš {len(df['label'].unique())} klasių")
df.head()

In [ ]:
df.describe()

In [ ]:
# ========================== MOKYMAS ==========================
X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=200, max_depth=9,
                             random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# ========================== REZULTATAI ==========================
y_pred = clf.predict(X_test)
print("🎯 Tikslumas:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Confusion matrix
plt.figure(figsize=(10,8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d',
            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.title("Confusion Matrix - Gintaro klasifikacija")
plt.show()

In [ ]:
!pip install lazypredict -q

In [ ]:
from lazypredict.Supervised import LazyClassifier

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)

print(models)   # ← graži lentelė su ~30 modelių rezultatais